# AURA Teaser Check (task YL-004B)

Checks an AURA opportunity teaser against its evidence pack and the approved AURA offer list. Every claim gets a verdict:
**Supported**, **Unsupported** or **Needs Human Verification**. Nothing is published or sent; a human approves every teaser.
This is the notebook for the *teaser* judge. The notebook `claim_check_colab.ipynb` is for story drafts and stays as it is.

**How to use:** run the cells from top to bottom. First the free checks, then one real teaser, then the nine test cases.

**Before you start:** the Colab secret `GEMINI_API_KEY` must exist (key icon in the left sidebar, "Notebook access" on). Fictional data only.

### Step 1. Get the code (branch `lab-step3`) and install the packages
If the folder already exists from an earlier session, the cell updates it, so you always run the newest rules.

In [ ]:
import os

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "lab-step3"            # the teaser check lives on this branch, not on main
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    # A private repository needs a token. A public one works without it.
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        clone_url = REPO_URL
    !git clone -q -b {BRANCH} {clone_url} {FOLDER}
    !git -C {FOLDER} remote set-url origin {REPO_URL}   # do not keep the token in the repo settings
else:
    !git -C {FOLDER} fetch -q origin {BRANCH}
    !git -C {FOLDER} checkout -q {BRANCH}
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
!git log --oneline -1
!pip install -q -r requirements.txt

### Step 2. Connect the model
Same test as in the other notebook. It should print `ready`. If the model is overloaded (503), the cell waits and tries again.

In [ ]:
import os
from google.colab import userdata

# Read the key from Colab Secrets
try:
    os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError(
        "Secret GEMINI_API_KEY not found. Add it with the key icon in the left sidebar, "
        "switch on 'Notebook access', and run this cell again."
    )

os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini through the OpenAI-compatible API
os.environ["MODEL_NAME"] = "gemini-3.5-flash-lite"   # the model that worked in the research agent; change if needed

# Send one short message to check that the key and the model work. Waits and tries again when the model is busy.
import time
from openai import OpenAI, APIStatusError

client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
for attempt in range(5):
    try:
        reply = client.chat.completions.create(
            model=os.environ["MODEL_NAME"],
            messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        )
        print(reply.choices[0].message.content)
        break
    except APIStatusError as error:
        # 503 = the model is overloaded, 429 = per-minute limit. Both are temporary (404 would mean a wrong model name).
        if error.status_code in (429, 500, 502, 503, 504) and attempt < 4:
            wait = 20 * (attempt + 1)
            print(f"Model answered {error.status_code}, waiting {wait} s ...")
            time.sleep(wait)
        else:
            raise

### Step 3. Free checks (no model call)
The offline tests should end with `29 tests passed`. The dry run uses only the checks done in plain Python, so its result is **not** a verdict.

In [ ]:
%cd /content/claim-check
!python tests/test_teaser_check.py
!python -m teaser_check --suite cases_teaser_approved/cases.json --dry-run --out results/teaser_dryrun.csv

### Step 4. One real teaser with the model
The teaser AURA wrote on 3 Nov for the fictional Test Firma Demo GmbH. To check another teaser, save its text as a `.txt` file in Colab and put its path in `TEASER`.

In [ ]:
%cd /content/claim-check
import os, datetime
TEASER = "cases_teaser_approved/a0_aura_real.txt"
PACK = "packs/pack_Test_001_approved.json"
out = f"results/teaser_a0_{os.environ['MODEL_NAME']}_{datetime.date.today()}.csv"
!python -m teaser_check {PACK} {TEASER} --out {out}

### Step 5. All nine test cases
A0 is the real teaser (control), A1-A8 each have one planted error. A6 and A8 can only be found by the model. The output shows what was found and what was expected. Run it 3 times to see how stable the judge is, and keep the saved files.

In [ ]:
%cd /content/claim-check
import os, datetime
out = f"results/teaser_approved_{os.environ['MODEL_NAME']}_{datetime.date.today()}.csv"
!python -m teaser_check --suite cases_teaser_approved/cases.json --out {out}

### Step 6. Look at the results
Shows every row that is not plain "Supported".

In [ ]:
%cd /content/claim-check
import glob, pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

latest = sorted(glob.glob("results/teaser_approved_*.csv"))[-1]
print(latest)
results = pd.read_csv(latest)
print(results.groupby(["case", "verdict"], dropna=False).size().unstack(fill_value=0))
results[results["verdict"] != "Supported"][["case", "kind", "text", "verdict", "finding", "reason"]]

### Step 7. Save the results
Everything in Colab is deleted when the session ends. This cell downloads the results as a zip file.

In [ ]:
%cd /content/claim-check
!zip -qr /content/teaser_check_results.zip results
from google.colab import files
files.download("/content/teaser_check_results.zip")